In [ ]:
import pandas as pd
import imagehash
from PIL import Image
pd.set_option("display.max_colwidth", 100)

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(ROOT))

from Utils.project_utils import get_project_root

In [3]:
PROJECT_ROOT = get_project_root()
CLASSES_DIR = PROJECT_ROOT / "Classes"
DATA_DIR = PROJECT_ROOT / "Data"

### Prepare Metadata CSV

In [4]:
# Some paths have depth of 4 instead of 3, need to list them all out
files = [f for f in CLASSES_DIR.rglob("*") if f.is_file() and not f.name.startswith(".")]
print("Total files found:", len(files))

Total files found: 102713


In [5]:
data = []

In [6]:
# This cell may take a long time to run, to speed up, we can directly read from the processed mushroom_paths.csv file
for f in files:
    try:
        with Image.open(f) as img:
            w, h = img.size
            img_hash = imagehash.phash(img)
    except Exception:
        continue
    # file_path stored relative to CLASSES_DIR (i.e., relative to "Classes/")
    rel = f.relative_to(CLASSES_DIR).as_posix()

    data.append(
        [
            rel,
            len(Path(rel).parts),
            f.stem,
            f.suffix.lstrip("."),
            w,
            h,
            w * h,
            str(img_hash),
        ]
    )

df = pd.DataFrame(
    data,
    columns=["file_path", "depth", "file_name", "file_ext", "width", "height", "num_pixels", "phash"]
)

df["aspect_ratio"] = df["width"] / df["height"]

In [7]:
# This above cell may take a long time to run, to speed up, we can directly read from the processed mushroomPaths.csv file
CSV_PATH = DATA_DIR / "mushroomPaths.csv"

if len(data) != 0:
    df.to_csv(CSV_PATH, index=False)
else:
    df = pd.read_csv(CSV_PATH)

In [8]:
df.shape

(102711, 9)

In [9]:
df.head()

,file_path,depth,file_name,file_ext,width,height,num_pixels,phash,aspect_ratio
0,conditionally_edible/horn_of_plenty/8.png,3,8,png,512,512,262144,abb9d4a0aae5c4d4,1.0
1,conditionally_edible/horn_of_plenty/9.png,3,9,png,512,512,262144,a581b66e5c2d6b32,1.0
2,conditionally_edible/horn_of_plenty/14.png,3,14,png,512,512,262144,d8f40799f31ea870,1.0
3,conditionally_edible/horn_of_plenty/12.png,3,12,png,512,512,262144,a8a8e950d4af3f58,1.0
4,conditionally_edible/horn_of_plenty/13.png,3,13,png,512,512,262144,8e86638cdcad5bd0,1.0


In [10]:
print("File Depth Distribution:")
print(df.groupby('depth').size())

File Depth Distribution:
depth
3    95998
4     6713
dtype: int64


In [11]:
print("\nSample files with depth 3:")
df[df['depth'] == 3].head()


Sample files with depth 3:


,file_path,depth,file_name,file_ext,width,height,num_pixels,phash,aspect_ratio
0,conditionally_edible/horn_of_plenty/8.png,3,8,png,512,512,262144,abb9d4a0aae5c4d4,1.0
1,conditionally_edible/horn_of_plenty/9.png,3,9,png,512,512,262144,a581b66e5c2d6b32,1.0
2,conditionally_edible/horn_of_plenty/14.png,3,14,png,512,512,262144,d8f40799f31ea870,1.0
3,conditionally_edible/horn_of_plenty/12.png,3,12,png,512,512,262144,a8a8e950d4af3f58,1.0
4,conditionally_edible/horn_of_plenty/13.png,3,13,png,512,512,262144,8e86638cdcad5bd0,1.0


In [12]:
print("\nSample files with depth 4:")
df[df['depth'] == 4].head()


Sample files with depth 4:


,file_path,depth,file_name,file_ext,width,height,num_pixels,phash,aspect_ratio
20454,conditionally_edible/Mushrooms/Agaricus/251_J9uSJkkBULQ.jpg,4,251_J9uSJkkBULQ,jpg,800,531,424800,cedf756430330339,1.506591
20455,conditionally_edible/Mushrooms/Agaricus/357__w_XUQZMZEw.jpg,4,357__w_XUQZMZEw,jpg,800,600,480000,c6d879863d364b29,1.333333
20456,conditionally_edible/Mushrooms/Agaricus/374_Xap1L15Z8BM.jpg,4,374_Xap1L15Z8BM,jpg,777,600,466200,c6966b613cc68dcc,1.295000
20457,conditionally_edible/Mushrooms/Agaricus/442_bdfUNhioT1A.jpg,4,442_bdfUNhioT1A,jpg,800,600,480000,90926f2e7865cdcc,1.333333
20458,conditionally_edible/Mushrooms/Agaricus/092_YzaMvFvqkiM.jpg,4,092_YzaMvFvqkiM,jpg,800,575,460000,845f39addeb18216,1.391304


In [13]:
df[df['depth'] == 4]['file_path'].apply(lambda x: x.split('/')[:-1]).value_counts().head()

file_path
[conditionally_edible, Mushrooms, Lactarius]      1563
[conditionally_edible, Mushrooms, Russula]        1147
[conditionally_edible, Mushrooms, Boletus]        1073
[conditionally_edible, Mushrooms, Cortinarius]     836
[conditionally_edible, Mushrooms, Amanita]         750
Name: count, dtype: int64

`Mushrooms` is redundant in the paths; the directory immediately after `Mushrooms/` represents the species label.

In [14]:
df["class"] = df["file_path"].apply(lambda x: x.split("/")[0])
df["species"] = df.apply(
    lambda x: (
        x["file_path"].split("/")[2]
        if x["depth"] == 4
        else x["file_path"].split("/")[1]
    ),
    axis=1,
)

In [15]:
df[df['depth'] == 3].head()

,file_path,depth,file_name,file_ext,width,height,num_pixels,phash,aspect_ratio,class,species
0,conditionally_edible/horn_of_plenty/8.png,3,8,png,512,512,262144,abb9d4a0aae5c4d4,1.0,conditionally_edible,horn_of_plenty
1,conditionally_edible/horn_of_plenty/9.png,3,9,png,512,512,262144,a581b66e5c2d6b32,1.0,conditionally_edible,horn_of_plenty
2,conditionally_edible/horn_of_plenty/14.png,3,14,png,512,512,262144,d8f40799f31ea870,1.0,conditionally_edible,horn_of_plenty
3,conditionally_edible/horn_of_plenty/12.png,3,12,png,512,512,262144,a8a8e950d4af3f58,1.0,conditionally_edible,horn_of_plenty
4,conditionally_edible/horn_of_plenty/13.png,3,13,png,512,512,262144,8e86638cdcad5bd0,1.0,conditionally_edible,horn_of_plenty


In [16]:
df[df['depth'] == 4].head()

,file_path,depth,file_name,file_ext,width,height,num_pixels,phash,aspect_ratio,class,species
20454,conditionally_edible/Mushrooms/Agaricus/251_J9uSJkkBULQ.jpg,4,251_J9uSJkkBULQ,jpg,800,531,424800,cedf756430330339,1.506591,conditionally_edible,Agaricus
20455,conditionally_edible/Mushrooms/Agaricus/357__w_XUQZMZEw.jpg,4,357__w_XUQZMZEw,jpg,800,600,480000,c6d879863d364b29,1.333333,conditionally_edible,Agaricus
20456,conditionally_edible/Mushrooms/Agaricus/374_Xap1L15Z8BM.jpg,4,374_Xap1L15Z8BM,jpg,777,600,466200,c6966b613cc68dcc,1.295000,conditionally_edible,Agaricus
20457,conditionally_edible/Mushrooms/Agaricus/442_bdfUNhioT1A.jpg,4,442_bdfUNhioT1A,jpg,800,600,480000,90926f2e7865cdcc,1.333333,conditionally_edible,Agaricus
20458,conditionally_edible/Mushrooms/Agaricus/092_YzaMvFvqkiM.jpg,4,092_YzaMvFvqkiM,jpg,800,575,460000,845f39addeb18216,1.391304,conditionally_edible,Agaricus


In [17]:
df['class'].value_counts(normalize=True)

class
conditionally_edible    0.603314
poisonous               0.205801
edible                  0.179299
deadly                  0.011586
Name: proportion, dtype: float64

In [ ]:
df.to_csv(DATA_DIR / "mushroomDetailedPaths.csv", index=False)